# 3.8 · 图像特征基础 / Image Feature Basics

> **课程定位 / Where this fits**
> **Part 3 第 8 课**。最后一个数据类型：图像。在 CNN（Part 10）统治之前, 计算机视觉靠**手工特征**——本课讲清像素表示、HOG、颜色直方图。理解它们能让你明白 **CNN 到底自动学到了什么**, 也为小数据/无 GPU 场景提供轻量方案。
> Before CNNs ruled vision, hand-crafted features did the work. Understanding them reveals what CNNs automate, and offers a lightweight path for small-data/no-GPU settings.

> 💡 **面试相关 / Interview-relevant**
> - "图像怎么变成特征向量" ★★★（基础）
> - "为什么原始像素不是好特征" ★★★★（平移/光照不变性）
> - "HOG 的思想" ★★★
> - "传统 CV 特征 vs CNN" ★★★

---

## 学习目标 / Learning Objectives
1. 理解图像的**张量表示**（H×W×C）和原始像素作特征的**三个弱点**。
2. 掌握 **HOG**（方向梯度直方图）的思想与实现。
3. 用**颜色直方图**做颜色不变的特征。
4. 对比**手工特征 + 经典 ML vs 端到端 CNN** 的取舍。

## 目录 / TOC
1. [图像 = 张量 + 原始像素的弱点 ⭐](#1)
2. [🔢 数据：手写数字 Digits](#2)
3. [原始像素作特征（基线）](#3)
4. [HOG：方向梯度直方图 ⭐](#4)
5. [颜色直方图](#5)
6. [手工特征 vs CNN](#6)
7. [小结 + Part 3 半程回顾](#7)


<a id="1"></a>
## 1. 图像 = 张量 + 原始像素的弱点 ⭐ / Images as Tensors

图像在计算机里是**数值张量**：
- 灰度图: `(H, W)` 矩阵, 每个值 0-255 (像素亮度)
- 彩色图: `(H, W, 3)` 张量, 3 个通道 RGB
- 一批图: `(N, H, W, C)` —— 这就是 0.2 节 ndarray 的 4 维版

**为什么原始像素是糟糕的特征**（三个弱点）：

| 弱点 | 说明 |
|---|---|
| **维度爆炸** | 一张 224×224×3 彩图 = 15 万维。表格 ML 直接崩 |
| **无平移不变性** | 同一只猫左移 5 像素 → 每个像素值全变 → 模型认为是完全不同的输入 |
| **无光照/尺度不变性** | 调亮一点 → 所有像素 +20 → 模型懵 |

**好特征应该对"无关变化"不变, 对"语义差异"敏感**。HOG/颜色直方图就是朝这个方向的手工设计; CNN 则是**自动学**这种不变性。
Good features are invariant to nuisance changes, sensitive to semantic ones. HOG hand-designs this; CNNs learn it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
rng = np.random.default_rng(42)

digits = load_digits()
print(f"图像数: {len(digits.images)}")
print(f"单张图: {digits.images[0].shape} (8×8 灰度, 值 0-16)")
print(f"展平后: {digits.data.shape[1]} 维像素向量")

# 看几张 / show a few
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axes, digits.images[:8], digits.target[:8]):
    ax.imshow(img, cmap="gray_r"); ax.set_title(f"label: {lab}"); ax.axis("off")
plt.suptitle("手写数字 (8×8 像素)", y=1.1); plt.tight_layout(); plt.show()


<a id="3"></a>
## 3. 原始像素作特征（基线）/ Raw Pixels as Baseline

把 8×8 图展平成 64 维向量, 直接喂分类器。**小图能work, 但不是好做法**（无不变性）。


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_raw, y = digits.data, digits.target   # (1797, 64) 像素向量

acc_raw = cross_val_score(
    make_pipeline(StandardScaler(), SVC()), X_raw, y, cv=5).mean()
print(f"原始像素 (64维) + SVM: CV 准确率 = {acc_raw:.1%}")
print("8×8 小图上像素法还能用 — 但放到 224×224 真实图像就崩 (15万维 + 无不变性)")

# 演示平移不变性问题: 把一张图右移1像素, 像素向量距离 / shift breaks pixel features
img = digits.images[0].copy()
shifted = np.roll(img, 1, axis=1)        # 右移 1 像素
dist = np.linalg.norm(img.ravel() - shifted.ravel())
print(f"\n同一张图右移1像素后, 像素向量欧氏距离 = {dist:.1f}")
print(f"  (对比: 这张图和另一张不同数字的距离 = {np.linalg.norm(img.ravel()-digits.images[1].ravel()):.1f})")
print("  右移这么小的变化就产生了可观距离 → 像素特征对平移极不鲁棒")


<a id="4"></a>
## 4. HOG：方向梯度直方图 ⭐ / Histogram of Oriented Gradients

**经典 CV 的明星特征**（行人检测的奠基, Dalal & Triggs 2005）。

**核心思想**：物体的形状由**边缘的方向分布**决定, 而非具体像素值。HOG 步骤：
1. 算每个像素的**梯度**（亮度变化的方向和大小）—— 0.8 节的梯度在图像上的应用
2. 把图分成小**cell**, 每个 cell 统计内部梯度方向的**直方图**
3. 拼接所有 cell 的直方图 → 特征向量

**为什么更好**：梯度对**光照变化鲁棒**（加亮度不改变梯度方向）, 局部直方图对**小平移鲁棒**。
Gradients are robust to lighting (brightening doesn't change edge direction); local histograms tolerate small shifts.


In [ ]:
from skimage.feature import hog
from skimage import exposure

# 对一张数字图算 HOG / compute HOG on one digit
img = digits.images[0]
features, hog_image = hog(img, orientations=8, pixels_per_cell=(2,2),
                          cells_per_block=(1,1), visualize=True)
print(f"8×8 图 → HOG 特征 {len(features)} 维")

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap="gray_r"); axes[0].set_title("原始数字"); axes[0].axis("off")
hog_vis = exposure.rescale_intensity(hog_image, in_range=(0, hog_image.max()))
axes[1].imshow(hog_vis, cmap="gray"); axes[1].set_title("HOG 可视化 (边缘方向)"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print("HOG 图显示了笔画的边缘方向 — 这才是'数字形状'的本质, 不是逐像素亮度")


In [ ]:
# HOG 特征 + 分类器, 对比原始像素 / HOG features vs raw pixels
X_hog = np.array([hog(img, orientations=8, pixels_per_cell=(2,2), cells_per_block=(1,1))
                  for img in digits.images])
print(f"全部图的 HOG 特征矩阵: {X_hog.shape}")

acc_hog = cross_val_score(make_pipeline(StandardScaler(), SVC()), X_hog, y, cv=5).mean()
print(f"\n原始像素 + SVM: {acc_raw:.1%}")
print(f"HOG 特征 + SVM: {acc_hog:.1%}")
print("HOG 用边缘方向信息, 在这个小数据上和像素接近; 真实场景(光照/平移变化大)优势明显")


<a id="5"></a>
## 5. 颜色直方图 / Color Histograms

**最简单的图像特征之一**：统计图像中各颜色/亮度的**分布**, 完全**丢弃空间位置**。

**适用**：颜色是关键判别信息的任务（区分蓝天 vs 草地、成熟果实 vs 未熟）, 且**对平移/旋转完全不变**（位置信息全丢了）。**不适用**：形状/纹理重要的任务（数字识别——颜色直方图分不出 3 和 8）。
Color histograms discard all spatial info — great when color discriminates (sky vs grass), useless when shape matters (3 vs 8).


In [ ]:
# 亮度直方图 (灰度图的"颜色"直方图) / intensity histogram
fig, axes = plt.subplots(2, 4, figsize=(13, 5))
for col, dig in enumerate([0, 1, 8, 7]):
    idx = np.where(digits.target == dig)[0][0]
    axes[0, col].imshow(digits.images[idx], cmap="gray_r")
    axes[0, col].set_title(f"digit {dig}"); axes[0, col].axis("off")
    hist, _ = np.histogram(digits.images[idx].ravel(), bins=17, range=(0, 16))
    axes[1, col].bar(range(17), hist); axes[1, col].set_title("亮度直方图")
plt.tight_layout(); plt.show()
print("注意: 0 和 8 的亮度直方图相似(都有很多笔画) → 颜色/亮度直方图分不开形状相近的数字")
print("→ 形状任务用 HOG, 颜色任务用颜色直方图 — 特征要匹配任务")


<a id="6"></a>
## 6. 手工特征 vs CNN / Hand-crafted vs CNN

| | 手工特征 (HOG/SIFT/颜色直方图) + 经典 ML | 端到端 CNN (Part 10) |
|---|---|---|
| 特征来源 | **人设计** (基于领域知识) | **数据自动学** |
| 数据需求 | 少 | 大 (或预训练迁移) |
| 算力 | CPU 可跑 | 需 GPU |
| 不变性 | 手动设计 (HOG 对光照) | 自动学 (卷积+池化) |
| 性能上限 | 中 | **高** (ImageNet 碾压传统方法) |
| 可解释 | 较好 | 黑盒 |
| 何时用 | **小数据 / 无 GPU / 需可解释 / 简单任务** | 大数据 / 复杂视觉任务 |

**历史转折点 = 2012 AlexNet**：在 ImageNet 上, CNN 自动学的特征首次大幅碾压几十年积累的手工特征——计算机视觉从此进入深度学习时代。但**理解 HOG 仍有价值**：它揭示了 CNN 早期卷积层学到的正是类似"边缘方向"的东西。
The 2012 AlexNet moment: learned features crushed decades of hand-crafted ones. But HOG still matters — early CNN layers learn exactly these edge-direction features.


In [ ]:
# 一个直观证据: CNN 第一层卷积核 ≈ 边缘检测器 ≈ HOG 的思想
# (这里用 Sobel 算子演示"边缘方向"提取, CNN 自动学的就是类似的核)
from scipy import ndimage
img = digits.images[0].astype(float)
sobel_x = ndimage.sobel(img, axis=1)    # 水平梯度 / horizontal edges
sobel_y = ndimage.sobel(img, axis=0)    # 垂直梯度 / vertical edges

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img, cmap="gray_r"); axes[0].set_title("原图"); axes[0].axis("off")
axes[1].imshow(np.abs(sobel_x), cmap="gray"); axes[1].set_title("水平边缘 (Sobel-x)"); axes[1].axis("off")
axes[2].imshow(np.abs(sobel_y), cmap="gray"); axes[2].set_title("垂直边缘 (Sobel-y)"); axes[2].axis("off")
plt.tight_layout(); plt.show()
print("Sobel 边缘检测 = HOG 的梯度步骤 = CNN 第一层卷积核学到的东西")
print("理解传统特征 → 理解深度学习在自动化什么")


<a id="7"></a>
## 7. 小结 + Part 3 半程回顾 / Summary

```
图像 = 张量 (H×W×C); 原始像素弱点: 维度爆炸 + 无平移/光照不变性
手工特征:
  HOG — 梯度方向直方图; 对光照鲁棒, 抓形状 ⭐ (行人检测奠基)
  颜色直方图 — 丢弃空间, 抓颜色分布; 平移旋转全不变
  特征要匹配任务: 形状→HOG, 颜色→颜色直方图
手工特征 + 经典ML vs CNN: 小数据/CPU/可解释 vs 大数据/GPU/高性能
2012 AlexNet: 学习特征碾压手工特征; 但 CNN 早期层学的正是边缘(=HOG)
```

### 💡 面试速查
1. **原始像素弱点**: 维度爆炸 + 无平移/光照不变性
2. **HOG**: 梯度方向直方图, 对光照鲁棒
3. **颜色直方图**: 丢空间信息, 颜色任务用, 形状任务别用
4. **CNN 自动学**手工特征曾经手动设计的不变性

### 数据类型四部曲完成
3.4-3.5 数值/类别 → 3.6 特征工程 → 3.7 文本 → 3.8 图像。**四种主要数据类型的特征化都覆盖了**。

### 下一节
**3.9 数据泄漏**——前面反复提"防泄漏"(填补/缩放/编码/聚合都只 fit train), 这一课**集中火力**讲清泄漏的所有形态和防御。
